## This notebook enables you to run the local Llama3.1-8B model on Kaggle's GPU free of charge. You can casually chat with the model and conduct simple experiments.

To get started, you need to create a Huggingface account, agree to the license agreement with Meta, and generate an access token. If you'd prefer not to do this, I recommend using my [Qwen 3 notebook](https://www.kaggle.com/code/cosmicwanderer2/free-qwen3-8b-on-kaggle-with-gpu instead.

In [ ]:
!pip install bitsandbytes

In [1]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="meta-llama/Llama-3.1-8B-Instruct", local_dir='/llama')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(
    '/llama',
    quantization_config = quantization_config,
    device_map = 'cuda:0',
    dtype = torch.float16,
    local_files_only=True)

tokenizer = AutoTokenizer.from_pretrained(
    '/llama',
    local_files_only=True
)

In [ ]:
from transformers import pipeline

pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

messages = [
    {"role": "system", "content": "You are a chatbot that provides clear, concise, short answer that can be understood by everybody."},
]

In [ ]:
MAX_MESSAGES = 5 
while True:
    user_input = input("You: ")
    if user_input.lower() in ["bye", "quit", "exit"]:
        break

    messages.append({"role": "user", "content": user_input})
    messages = [messages[0]] + messages[1:][-5:]
    outputs = pipeline(messages)
    response = outputs[0]["generated_text"][-1]['content']
    print(f"Llama: {response}")
    messages.append({"role": "assistant", "content": response})